# Part 3b — Learning by doing

### Two channels, and neither of them changes the plan much

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/lithium-modelling/blob/main/notebooks/03b_production_learning.ipynb)

Part 3 had one learning channel: build more capacity, and capacity gets cheaper.
That is learning-by-*building*. It is also only half of what the literature means
by a learning curve — the other half is learning-by-*doing*, where cost falls
because you have made a lot of the stuff, not because you have built a lot of
factories.

| | driver | what gets cheaper | how it is modelled |
|---|---|---|---|
| **Channel A** | cumulative **capacity** | capex | SOS2 on a continuous curve (Part 3 §9) |
| **Channel B** | lagged cumulative **production** | opex | discrete tiers with binaries (§6) |

### Why Channel B is discrete when Channel A is continuous

Not for realism — for **linearity**. Channel A multiplies a curve by nothing: the
model reads a cumulative cost off it and pays the difference. Channel B has to
multiply an opex *multiplier* by a *throughput*, and both are variables. That
product is bilinear and a MILP cannot have it.

Tiers solve it: a binary picks which multiplier applies, throughput is split
across tiers, and multiplier × throughput becomes a sum of constants × variables.
§6 builds that, and it costs 234 extra binaries.

### The three questions this notebook answers

1. **Do the two channels interfere?** No — they are separable to the last
   decimal, and §8 asserts it.
2. **Would a planner overproduce just to learn faster?** No, and §11 fails to
   make it happen even with free disposal and a 55% learning rate.
3. **Does any of it change the build plan?** Barely. §13.

### A note on the lag

Know-how does not arrive the instant a unit is made. `LAG_YEARS = 3` delays it —
and the lag is defined in **years**, then mapped to whichever period contains
that year. Defining it in periods instead would silently stretch from 3 years to
9 as the mesh coarsens, which is the kind of bug that produces a plausible answer.

## 0. Setup

One cell, and it is the only place the `lithium` package appears before the final check. On Colab it
clones the repo and installs it; locally it assumes you have already run `pip install -e .` and just
moves up out of `notebooks/` so the relative data paths work.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/USERNAME/lithium-modelling.git"   # <-- edit me
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

if "google.colab" in sys.modules:
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"font.size": 12, "axes.grid": True, "grid.alpha": 0.3})
print(f"working directory : {Path.cwd().name}")
print(f"gurobipy          : {gp.gurobi.version()}")
print(f"pandas            : {pd.__version__}")

working directory : Advanced Opt Modeling Examples
gurobipy          : (13, 0, 2)
pandas            : 2.3.3


## 2. The instance

**This is the fourth instance in the series, and it is the one most easily
confused with Part 4's**, because it has the same three stages and the same two
regions. The differences are the whole reason both exist:

| | here (Parts 3, 3b) | Part 4's chain |
|---|---|---|
| who decides | one planner, minimising cost | two firms, maximising profit |
| what crosses regions | anything, at any stage | finished goods only |
| learning | one industry-wide pool | per firm |
| price | none; demand must be served | endogenous, or fixed |
| costs by region | **symmetric** | asymmetric |

The symmetric costs matter. With both regions identical on cost, whatever
asymmetry appears in the answer came from demand, legacy fleets and geography —
not from one region simply being cheaper. That makes this the right place to
study the machinery before Part 4 adds a thumb to the scale.

| file | keyed by | rows |
|---|---|---|
| `netcore_stages.csv` | `stage` | 3 |
| `netcore_nodes.csv` | `(stage, region)` | 6 |
| `netcore_regions.csv` | `region` | 2 |
| `efficiency.csv` | `stage` | 3 |

`efficiency.csv` is **the same file Part 4 reads**, not a copy of it. The yield
model is genuinely shared; the cost tables are not.

In [2]:
DATA = Path("data/raw")

if not (DATA / "netcore_stages.csv").exists():
    print("!" * 78)
    print("! data/raw/ was not found, so this notebook is FALLING BACK to generated")
    print("! numbers. Everything below will run and every figure will render, but the")
    print("! results are NOT the shipped instance and are NOT an acceptable submission.")
    print("! Fix: clone the repo (see section 0) or run this notebook from the repo root.")
    print("!" * 78)
    DATA = Path("_generated_fallback")
    DATA.mkdir(exist_ok=True)
    (DATA / "netcore_stages.csv").write_text(
        "stage,fixed,unit,operate,lead\nMINE,900.0,7.0,1.2,1\n"
        "PROC,1500.0,11.0,2.0,2\nMFG,1300.0,9.5,2.4,2\n")
    (DATA / "netcore_nodes.csv").write_text(
        "stage,region,legacy_cap,legacy_ret\nMINE,R1,200,9\nMINE,R2,180,14\n"
        "PROC,R1,170,12\nPROC,R2,150,19\nMFG,R1,130,16\nMFG,R2,120,24\n")
    (DATA / "netcore_regions.csv").write_text(
        "region,demand_base,demand_growth\nR1,100.0,0.008\nR2,70.0,0.026\n")
    (DATA / "efficiency.csv").write_text(
        "stage,eta_ceil,eta_base,alpha,beta,delta_bar\nMINE,0.92,0.86,0.0,0.0,0.02\n"
        "PROC,0.95,0.80,0.030,0.010,0.05\nMFG,0.93,0.78,0.025,0.008,0.05\n")

stages_df = pd.read_csv(DATA / "netcore_stages.csv")
nodes_df = pd.read_csv(DATA / "netcore_nodes.csv")
regions_df = pd.read_csv(DATA / "netcore_regions.csv")
eff_df = pd.read_csv(DATA / "efficiency.csv")
for nm, df in (("netcore_stages", stages_df), ("netcore_nodes", nodes_df),
               ("netcore_regions", regions_df), ("efficiency", eff_df)):
    print(f"{nm + '.csv':22s} {len(df)} rows x {len(df.columns)} columns")
stages_df

netcore_stages.csv     3 rows x 5 columns
netcore_nodes.csv      6 rows x 4 columns
netcore_regions.csv    2 rows x 3 columns
efficiency.csv         3 rows x 6 columns


,stage,fixed,unit,operate,lead
0,MINE,900.0,7.0,1.2,1
1,PROC,1500.0,11.0,2.0,2
2,MFG,1300.0,9.5,2.4,2


A frame shows rows and columns; the model indexes by `stage` and by
`(stage, region)`. Print the dictionary form so the **key** is explicit rather
than implied by the layout — every constraint below looks values up by one of
those two keys, and getting them confused is the most common way to build a
model that solves and means nothing.

In [3]:
STAGES = tuple(stages_df["stage"])
REGIONS = tuple(regions_df["region"])
NODES = [(s, r) for s in STAGES for r in REGIONS]
ARCS = [(s, a, b) for s in STAGES for a in REGIONS for b in REGIONS]

FIXED = dict(zip(stages_df["stage"], stages_df["fixed"].astype(float)))
UNIT = dict(zip(stages_df["stage"], stages_df["unit"].astype(float)))
OPERATE = dict(zip(stages_df["stage"], stages_df["operate"].astype(float)))
LEAD = dict(zip(stages_df["stage"], stages_df["lead"].astype(int)))

_k = list(zip(nodes_df["stage"], nodes_df["region"]))
LEGACY_CAP = dict(zip(_k, nodes_df["legacy_cap"].astype(float)))
LEGACY_RET = dict(zip(_k, nodes_df["legacy_ret"].astype(int)))

DEMAND_BASE = dict(zip(regions_df["region"], regions_df["demand_base"].astype(float)))
DEMAND_GROWTH = dict(zip(regions_df["region"], regions_df["demand_growth"].astype(float)))

ETA_CEIL = dict(zip(eff_df["stage"], eff_df["eta_ceil"].astype(float)))
ETA_BASE = dict(zip(eff_df["stage"], eff_df["eta_base"].astype(float)))
ALPHA = dict(zip(eff_df["stage"], eff_df["alpha"].astype(float)))
BETA = dict(zip(eff_df["stage"], eff_df["beta"].astype(float)))
DELTA_BAR = dict(zip(eff_df["stage"], eff_df["delta_bar"].astype(float)))

print(f"{len(NODES)} nodes: {NODES}")
print(f"{len(ARCS)} arcs, e.g. {ARCS[:3]} ...")
print(f"\n{'FIXED':12s} {FIXED}")
print(f"{'UNIT':12s} {UNIT}")
print(f"{'OPERATE':12s} {OPERATE}")
print(f"{'LEAD':12s} {LEAD}")
print(f"{'LEGACY_CAP':12s} {LEGACY_CAP}")
print(f"{'LEGACY_RET':12s} {LEGACY_RET}")

# Try it: make R2 the cheap region and re-run. Section 14 stays green,
# because the package is handed this table rather than re-reading the file.
# FIXED, UNIT, OPERATE are keyed by STAGE here, so that edit needs the model to
# index them by (stage, region) first - which is exactly what Part 4 does.

6 nodes: [('MINE', 'R1'), ('MINE', 'R2'), ('PROC', 'R1'), ('PROC', 'R2'), ('MFG', 'R1'), ('MFG', 'R2')]
12 arcs, e.g. [('MINE', 'R1', 'R1'), ('MINE', 'R1', 'R2'), ('MINE', 'R2', 'R1')] ...

FIXED        {'MINE': 900.0, 'PROC': 1500.0, 'MFG': 1300.0}
UNIT         {'MINE': 7.0, 'PROC': 11.0, 'MFG': 9.5}
OPERATE      {'MINE': 1.2, 'PROC': 2.0, 'MFG': 2.4}
LEAD         {'MINE': 1, 'PROC': 2, 'MFG': 2}
LEGACY_CAP   {('MINE', 'R1'): 200.0, ('MINE', 'R2'): 180.0, ('PROC', 'R1'): 170.0, ('PROC', 'R2'): 150.0, ('MFG', 'R1'): 130.0, ('MFG', 'R2'): 120.0}
LEGACY_RET   {('MINE', 'R1'): 9, ('MINE', 'R2'): 14, ('PROC', 'R1'): 12, ('PROC', 'R2'): 19, ('MFG', 'R1'): 16, ('MFG', 'R2'): 24}


**Every arc exists.** A mine in R1 can feed a processor in R2 and vice versa, at
every stage. That is the structural difference from Part 4, where each region's
chain is internally closed and only the finished product is traded, and it is
why this model has a `flow` variable indexed by `(stage, from, to, period)`
rather than a single sales variable.

## 3. Time, and the knobs everything hangs off

### Variable-length periods, and why

A 39-year horizon at annual resolution would be 39 periods and a model several
times the size. But the near years are where the decisions are, and the far
years are there to stop the model treating the end of the horizon as the end of
the world. So the mesh is **fine early and coarse late**.

`BLOCKS` is a list of `(how many periods, years in each)`. Everything downstream
— period starts, discount weights, which vintages are operating when — is
arithmetic on it, which is why it is a knob and not a table.

**`REPORT_UNTIL` is the other half of the same idea.** The horizon runs past the
last year anyone reads, so that a facility built in the reporting window still
captures most of its 25-year life *inside* the model. Without that buffer the
model refuses to build late, not because building late is bad but because the
model stops before the asset has paid for itself. Section 16 measures what that
buffer is worth.

In [4]:
BLOCKS = [(6, 1), (4, 3), (2, 5), (1, 9)]   # (how many periods, years in each)
DR = 0.05           # discount rate
LIFE = 25           # asset life, years
CAP_MIN, CAP_MAX = 60.0, 260.0    # a facility that is built is between these
LEGACY_BYR = -8     # inherited assets are already 8 years old
ETA_FLOOR = 0.60    # no vintage is ever worse than this
REPORT_UNTIL = 28   # years past this are the cool-down buffer

LEN, START, _y = [], [], 1
for _count, _length in BLOCKS:
    for _ in range(_count):
        LEN.append(_length)
        START.append(_y)
        _y += _length
P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}

# OMEGA[p] is the SUM of annual discount factors inside period p, so a
# per-year cost multiplied by it becomes that period's present value. It is NOT
# an average: a 9-year period must carry nine years of cost.
OMEGA = {p: sum(1 / (1 + DR) ** t for t in YEARS[p]) for p in P}

print(f"{len(P)} periods spanning {HORIZON} years; "
      f"sum of weights = {sum(OMEGA.values()):.3f}")
print(f"reporting window ends at year {REPORT_UNTIL}; "
      f"years {REPORT_UNTIL + 1}-{HORIZON} are the cool-down buffer")
pd.DataFrame([dict(period=p, years=f"{START[p]}-{START[p] + LEN[p] - 1}",
                   length=LEN[p], omega=round(OMEGA[p], 3)) for p in P])

13 periods spanning 37 years; sum of weights = 16.711
reporting window ends at year 28; years 29-37 are the cool-down buffer


,period,years,length,omega
0,0,1-1,1,0.952
1,1,2-2,1,0.907
2,2,3-3,1,0.864
3,3,4-4,1,0.823
4,4,5-5,1,0.784
5,5,6-6,1,0.746
6,6,7-9,3,2.032
7,7,10-12,3,1.755
8,8,13-15,3,1.516
9,9,16-18,3,1.310


### 3.1 The capital recovery factor, and charging capex by the year

A facility bought in year $v$ is paid for once but used for `LIFE` years. Charging
the whole cheque in year $v$ makes late builds look ruinous; charging nothing
makes them free. The standard fix is to **annuitise**: convert the lump sum into
an equivalent annual payment with the capital recovery factor, then discount only
the years that fall inside the horizon.

$$\text{CRF} = \frac{r(1+r)^{L}}{(1+r)^{L}-1}
\qquad\qquad
\mu_{s,v} = \text{CRF}\sum_{t=\text{online}}^{\text{online}+L-1}\!\!\!\!
\frac{\mathbb{1}[t \le T]}{(1+r)^{t}}$$

**`MU` is the coefficient that makes late builds behave sensibly**, and section
16 compares it against the lump-sum alternative.

In [5]:
CRF = DR * (1 + DR) ** LIFE / ((1 + DR) ** LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}

MU = {(s, v): CRF * sum(1 / (1 + DR) ** t
                        for t in range(ONLINE[s, v], ONLINE[s, v] + LIFE)
                        if t <= HORIZON)
      for s in STAGES for v in P}

print(f"CRF ({LIFE} yr at {DR:.0%}) = {CRF:.5f}")
print("MU for PROC by decision period:",
      {v: round(MU['PROC', v], 3) for v in [0, len(P) // 3, len(P) - 4, len(P) - 1]})
print("\nMU falls at the end of the horizon because a late build's later years")
print("fall outside it. That is the truncation the buffer is there to absorb.")

CRF (25 yr at 5%) = 0.07095
MU for PROC by decision period: {0: 0.907, 4: 0.746, 9: 0.386, 12: 0.095}

MU falls at the end of the horizon because a late build's later years
fall outside it. That is the truncation the buffer is there to absorb.


### 3.2 Demand

Each region's demand grows at its own rate. R2 starts smaller and grows faster,
which is what makes the siting question interesting rather than obvious.

In [6]:
DEMAND = {}
for r in REGIONS:
    base, g = DEMAND_BASE[r], DEMAND_GROWTH[r]
    for p in P:
        DEMAND[r, p] = sum(base * (1 + g) ** (t - 1) for t in YEARS[p]) / LEN[p]

TRANSPORT_OWN, TRANSPORT_CROSS = 0.5, 2.0
TRANSPORT = {(a, b): (TRANSPORT_OWN if a == b else TRANSPORT_CROSS)
             for a in REGIONS for b in REGIONS}
PEN_SHORT = 90.0     # per unit of unmet final demand

print(f"demand rate, year 1 : { {r: round(DEMAND[r, 0], 1) for r in REGIONS} }")
print(f"demand rate, year {START[P[-1]]}: "
      f"{ {r: round(DEMAND[r, P[-1]], 1) for r in REGIONS} }")
cross = [r for r in REGIONS if DEMAND[r, P[-1]] > DEMAND[REGIONS[0], P[-1]]]
print(f"\ncrossing regions costs {TRANSPORT_CROSS / TRANSPORT_OWN:.0f}x staying home")
print(f"unmet demand costs {PEN_SHORT}, which is "
      f"{PEN_SHORT / max(OPERATE.values()):.0f}x the dearest stage's opex")

demand rate, year 1 : {'R1': 100.0, 'R2': 70.0}
demand rate, year 29: {'R1': 129.1, 'R2': 159.5}

crossing regions costs 4x staying home
unmet demand costs 90.0, which is 38x the dearest stage's opex


### 3.3 Vintage efficiency: two channels, both indexed by vintage

Yield improves for two separate reasons and the model keeps them apart.

- **A later vintage starts better.** The frontier moves at rate $\alpha$ per
  year, so a facility built in year 20 begins life more efficient than one built
  in year 1 ever becomes.
- **An asset improves with age**, at rate $\beta$, but by at most
  $\bar\delta$ over its whole life. Retrofits help; they do not turn a 1990s
  plant into a new one.

Both are capped by the stage's ceiling $\bar\eta$ and floored at `ETA_FLOOR`.
The result is `ETA[stage, vintage, period]` — a yield that depends on **when it
was built and when it is running**, which is why throughput has to be tracked
per vintage rather than per node.

In [7]:
VINTAGES = [-1] + P          # -1 is the inherited legacy cohort
BUILD_YEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}

ETA = {}
for s in STAGES:
    for v in VINTAGES:
        frontier = ETA_CEIL[s] - (ETA_CEIL[s] - ETA_BASE[s]) \
            * (1 - ALPHA[s]) ** (BUILD_YEAR[v] - 1)
        frontier = max(ETA_FLOOR, min(frontier, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p] - BUILD_YEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s] - frontier) * (1 - BETA[s]) ** age
            ETA[s, v, p] = max(ETA_FLOOR, min(frontier + DELTA_BAR[s], aged))

# the theory, asserted: a yield is a fraction, never exceeds its ceiling, and a
# later vintage is never worse than an earlier one in the same period
for s in STAGES:
    for p in P:
        assert all(ETA_FLOOR - 1e-12 <= ETA[s, v, p] <= ETA_CEIL[s] + 1e-12
                   for v in VINTAGES), f"{s} yield left [floor, ceiling] in period {p}"
        vs = [v for v in P if v <= p]
        assert all(ETA[s, vs[i], p] <= ETA[s, vs[i + 1], p] + 1e-12
                   for i in range(len(vs) - 1)), \
            f"{s}: a later vintage is worse than an earlier one in period {p}"
print(f"{len(ETA)} yields, keyed (stage, vintage, period)")
print("\nPROC yield by vintage and operating period:")
_show_p = [0, len(P) // 3, 2 * len(P) // 3, len(P) - 1]
pd.DataFrame({f"vintage {v}": [round(ETA['PROC', v, p], 4) for p in _show_p]
              for v in [-1, 0, len(P) // 2]},
             index=[f"yr {START[p]}" for p in _show_p]).T

546 yields, keyed (stage, vintage, period)

PROC yield by vintage and operating period:


,yr 1,yr 5,yr 13,yr 29
vintage -1,0.7698,0.7769,0.7902,0.8027
vintage 0,0.8000,0.8059,0.8170,0.8368
vintage 6,0.8251,0.8251,0.8324,0.8498


### 3.4 Which vintages are operating when

A legacy asset runs until its retirement year, **inclusive**. A built asset runs
from `LEAD` years after its decision, for `LIFE` years. `ACTIVE` is the set of
`(stage, region, vintage, period)` combinations that exist at all, and `VIN` is
the same information indexed the way the constraints need it.

Building these sets explicitly, rather than writing `if` conditions inside every
constraint, is what keeps the model readable — and it means a mistake in the
timing shows up here as a wrong count rather than as a silently missing
constraint fifty lines further down.

In [8]:
ACTIVE = [(s, r, v, p)
          for (s, r) in NODES for v in VINTAGES for p in P
          if (v == -1 and START[p] <= LEGACY_RET[s, r])
          or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v] + LIFE - 1)]

VIN = {}
for (s, r, v, p) in ACTIVE:
    VIN.setdefault((s, r, p), []).append(v)

# a build decision is only meaningful if the asset comes online inside the horizon
BUILD = [(s, r, v) for (s, r) in NODES for v in P if ONLINE[s, v] <= HORIZON]

assert len(ACTIVE) > 0 and len(BUILD) > 0, "an empty index set reports success too"
print(f"{len(ACTIVE)} active (node, vintage, period) triples")
print(f"{len(BUILD)} candidate build decisions -> {len(BUILD)} binaries")
print(f"\nlegacy MINE/R1 retires in year {LEGACY_RET['MINE', 'R1']} and is active "
      f"in {sum(1 for (s, r, v, p) in ACTIVE if (s, r, v) == ('MINE', 'R1', -1))} periods")

487 active (node, vintage, period) triples
78 candidate build decisions -> 78 binaries

legacy MINE/R1 retires in year 9 and is active in 7 periods


## 4. Carried over from Part 3: Channel A

The capex learning curve is Part 3 §4 unchanged apart from two knobs — a gentler
learning rate and a higher floor. It arrives under a `CARRIED OVER` marker rather
than being re-narrated; if you have not read Part 3 §4 and §9, read them before
this cell.

The two knobs that differ, and they are knobs rather than corrections: Part 3
used `LR_CAPEX = 0.20` with a floor at 0.55, this notebook uses **0.15** and
**0.60**. A gentler capex channel makes the contrast with Channel B legible
rather than swamped.

In [9]:
# CARRIED OVER FROM 03_network_core SECTIONS 4 AND 9 - narrated there.
import math
import time

LEARN_STAGES = ["PROC", "MFG"]
LR_CAPEX = 0.15          # gentler than Part 3's 0.20
Q_START, Q_ADD = 400.0, 1000.0
CAPEX_FLOOR = 0.60       # higher floor than Part 3's 0.55
NBP = 9
PANELS = 600

_bc = -math.log2(1 - LR_CAPEX)
U0 = sum(UNIT[s] for s in LEARN_STAGES) / len(LEARN_STAGES)


def capex_unit(q):
    return max(CAPEX_FLOOR * U0, U0 * (q / Q_START) ** (-_bc))


def capex_cum(q, panels=PANELS):
    if q <= Q_START:
        return 0.0
    h = (q - Q_START) / panels
    return sum(0.5 * (capex_unit(Q_START + i * h) + capex_unit(Q_START + (i + 1) * h)) * h
               for i in range(panels))


K = list(range(NBP))
QBP = [Q_START + Q_ADD * k / (NBP - 1) for k in K]
CBP = [capex_cum(q) for q in QBP]
MU_TECH = {p: MU[LEARN_STAGES[0], p] for p in P}

print(f"Channel A: LR {LR_CAPEX:.0%}, floor {CAPEX_FLOOR:.0%}, {NBP} breakpoints")
print(f"unit cost across the mesh {[round(capex_unit(q), 2) for q in QBP]}")

Channel A: LR 15%, floor 60%, 9 breakpoints
unit cost across the mesh [10.25, 9.62, 9.15, 8.78, 8.48, 8.22, 8.0, 7.81, 7.64]


## 5. Channel B: the parameters, and what has to be calibrated

Opex falls with **cumulative production**, in `N_TIERS` discrete steps. Two
things need deciding before any of it can be built.

**Where do the thresholds go?** They cannot be guessed, and they cannot come from
a solve that already has production learning — that would be circular. So §7
solves the model *without* Channel B, observes how much each stage actually
produces, and places the thresholds at **doublings** of that. Doublings, because
Wright's law is stated per doubling: the multiplier at tier $j$ is then exactly
$(1 - \text{LR})^{j}$.

**What is the lag for?** Cumulative production at the *lagged* period drives the
current tier, because know-how takes time to embody in practice. Set
`LAG_YEARS = 0` and the model gets its discount the instant it produces, which is
both wrong and makes the tier constraints easier — a bad combination.

In [10]:
LR_OPEX = 0.18        # opex falls this much per doubling of cumulative production
OPEX_FLOOR = 0.65     # floor, as a fraction of base opex
LAG_YEARS = 3         # know-how embodies with a delay, defined in YEARS
N_TIERS = 3           # tier 0 is the no-discount tier
LEARN_SCOPE = "regional"   # 'regional' | 'global'

PEN_DISPOSE = 12.0    # cost of throwing product away
PEN_DEVIATE = 35.0    # one-sided penalty for undershooting a local-content floor
TIER_MIN_PHASE_IN = 6      # no local-content minimum binds before this year

MIPGAP = 1e-6
# 1e-6, not the 0.005 the original used. At 0.005 the capacity variant stopped
# at 45,547.7 here and 45,546.0 in an equivalent formulation - a difference of
# 3.6e-05, invisible against the gap but larger than several effects this
# notebook measures.

# the lag, in years, mapped to whichever period holds that year
LAGP = {p: YEAR_TO_P[max(1, START[p] - LAG_YEARS)] for p in P}
print(f"Channel B: LR {LR_OPEX:.0%}, floor {OPEX_FLOOR:.0%}, {N_TIERS} tiers, "
      f"{LAG_YEARS}-year lag, scope '{LEARN_SCOPE}'")
print(f"\nthe lag maps period -> lagged period: "
      f"{ {p: LAGP[p] for p in P[:6]} } ...")
print("defined in YEARS: period 8 starts in year "
      f"{START[8]} and looks back to year {max(1, START[8] - LAG_YEARS)}, "
      f"which is period {LAGP[8]}")
print("had the lag been '3 periods' it would mean "
      f"{START[8] - START[max(0, 8 - 3)]} years here and more later - "
      "a bug that produces plausible output")

Channel B: LR 18%, floor 65%, 3 tiers, 3-year lag, scope 'regional'

the lag maps period -> lagged period: {0: 0, 1: 0, 2: 0, 3: 0, 4: 1, 5: 2} ...
defined in YEARS: period 8 starts in year 13 and looks back to year 10, which is period 7
had the lag been '3 periods' it would mean 7 years here and more later - a bug that produces plausible output


## 6. The model, with both channels

The skeleton is Part 3's: semi-continuous sizing, vintage-indexed throughput,
cross-region arcs, demand service. Three things are added.

**Cumulative production**, `cumprod[s, scope, p]` — undiscounted, because
know-how accrues in physical units. It uses `LEN[q]`, the number of years in
period `q`, and never `OMEGA`. Discounting a knowledge stock would be a category
error and a very easy one to make when every other sum in the model is
discounted.

**The tier selection**, `z[s, scope, p, j]` — exactly one tier per node-period,
with big-M constraints saying the chosen tier is consistent with lagged
cumulative production.

**The throughput split**, `tsplit[s, r, p, j]` — the linearisation. Throughput is
divided across tiers, each piece is charged at its tier's multiplier, and a
binary forces all of it into the selected tier. Note it splits **node-level**
throughput, not per-vintage: the opex rate does not depend on vintage, so the two
are exactly equivalent and this one is far smaller.

Disposal and the local-content floor are also here, unused until §11 and §12.

In [11]:
# THE FUNCTION IS THE LESSON: the four variants in section 8 must differ ONLY in
# which channels are switched on. Writing the model out per variant would make a
# difference between two rows uninterpretable - a change in the objective, or a
# typo in a constraint, with no way to tell. Every block is narrated above.
def build_model(learning="production", tiers=None, tier_min=None,
                allow_dispose=True, pen_dispose=PEN_DISPOSE,
                pen_deviate=PEN_DEVIATE, mipgap=MIPGAP):
    """learning: 'none' | 'capacity' | 'production' | 'both'"""
    tier_min = dict(tier_min or {})
    m = gp.Model()
    m.Params.OutputFlag = 0
    m.Params.MIPGap = mipgap

    build = m.addVars(BUILD, vtype=GRB.BINARY, name="build")
    size = m.addVars(BUILD, lb=0.0, ub=CAP_MAX, name="size")
    thr = m.addVars(ACTIVE, lb=0.0, name="thr")
    flow = m.addVars(ARCS, P, lb=0.0, name="flow")
    short = m.addVars(REGIONS, P, lb=0.0, name="short")
    dev = m.addVars(NODES, P, lb=0.0, name="dev")
    disp = m.addVars(REGIONS, P, lb=0.0, name="disp")

    m.addConstrs(size[s, r, v] <= CAP_MAX * build[s, r, v] for (s, r, v) in BUILD)
    m.addConstrs(size[s, r, v] >= CAP_MIN * build[s, r, v] for (s, r, v) in BUILD)
    m.addConstrs(thr[s, r, v, p] <= (LEGACY_CAP[s, r] if v == -1 else size[s, r, v])
                 for (s, r, v, p) in ACTIVE)
    m.addConstrs(gp.quicksum(ETA[s, v, p] * thr[s, r, v, p] for v in VIN[s, r, p])
                 == flow.sum(s, r, "*", p) for (s, r) in NODES for p in P)
    for i, s in enumerate(STAGES):
        if i == 0:
            continue
        m.addConstrs(flow.sum(STAGES[i - 1], "*", r, p)
                     == gp.quicksum(thr[s, r, v, p] for v in VIN[s, r, p])
                     for r in REGIONS for p in P)
    # equality with a disposal slack: surplus has to go somewhere and be paid for
    m.addConstrs(flow.sum(STAGES[-1], "*", r, p) + short[r, p] - disp[r, p]
                 == DEMAND[r, p] for r in REGIONS for p in P)
    if not allow_dispose:
        m.addConstrs(disp[r, p] == 0 for r in REGIONS for p in P)
    if tier_min:
        m.addConstrs(gp.quicksum(thr[s, r, v, p] for v in VIN[s, r, p]) + dev[s, r, p]
                     >= tier_min.get((s, r, p), 0.0)
                     for (s, r) in NODES for p in P)

    # cumulative production: UNDISCOUNTED, in physical units, so LEN not OMEGA
    scope = ([(s, r) for (s, r) in NODES] if LEARN_SCOPE == "regional"
             else [(s, "ALL") for s in STAGES])
    cum_ub = 3.0 * CAP_MAX * HORIZON * (len(REGIONS) if LEARN_SCOPE == "global" else 1)
    cumprod = m.addVars(scope, P, lb=0.0, ub=cum_ub, name="cumprod")
    m.addConstrs(cumprod[s, rk, p]
                 == gp.quicksum(LEN[q] * thr[s, r, v, q]
                                for r in (REGIONS if rk == "ALL" else [rk])
                                for q in P if q <= p for v in VIN[s, r, q])
                 for (s, rk) in scope for p in P)

    capex = (gp.quicksum(MU[s, v] * FIXED[s] * build[s, r, v] for (s, r, v) in BUILD)
             + gp.quicksum(MU[s, v] * UNIT[s] * size[s, r, v]
                           for (s, r, v) in BUILD if s not in LEARN_STAGES))
    if learning in ("capacity", "both"):
        Q = m.addVars(P, lb=Q_START, ub=Q_START + Q_ADD)
        Cc = m.addVars(P, lb=0.0)
        lam = m.addVars(P, K, lb=0.0, ub=1.0)
        m.addConstrs(lam.sum(p, "*") == 1 for p in P)
        m.addConstrs(Q[p] == gp.quicksum(QBP[k] * lam[p, k] for k in K) for p in P)
        m.addConstrs(Cc[p] == gp.quicksum(CBP[k] * lam[p, k] for k in K) for p in P)
        m.addConstrs(Q[p] == Q_START + gp.quicksum(size[s, r, v] for (s, r, v) in BUILD
                                                   if s in LEARN_STAGES and v <= p)
                     for p in P)
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])
        capex += gp.quicksum(MU_TECH[p] * (Cc[p] - (Cc[p - 1] if p > 0 else 0.0))
                             for p in P)
    else:
        capex += gp.quicksum(MU[s, v] * UNIT[s] * size[s, r, v]
                             for (s, r, v) in BUILD if s in LEARN_STAGES)

    if learning in ("production", "both") and tiers:
        tq, tm = tiers
        J = list(range(N_TIERS))
        z = m.addVars(scope, P, J, vtype=GRB.BINARY, name="tier")
        m.addConstrs(z.sum(s, rk, p, "*") == 1 for (s, rk) in scope for p in P)
        bigq = cum_ub
        m.addConstrs(cumprod[s, rk, LAGP[p]] >= tq[s][j - 1] - bigq * (1 - z[s, rk, p, j])
                     for (s, rk) in scope for p in P for j in J if j > 0)
        m.addConstrs(cumprod[s, rk, LAGP[p]] <= tq[s][j] + bigq * (1 - z[s, rk, p, j])
                     for (s, rk) in scope for p in P for j in J if j < N_TIERS - 1)
        tsplit = m.addVars(NODES, P, J, lb=0.0, name="tsplit")
        m.addConstrs(tsplit.sum(s, r, p, "*")
                     == gp.quicksum(thr[s, r, v, p] for v in VIN[s, r, p])
                     for (s, r) in NODES for p in P)
        m.addConstrs(tsplit[s, r, p, j]
                     <= 3 * CAP_MAX * z[s, (r if LEARN_SCOPE == "regional" else "ALL"), p, j]
                     for (s, r) in NODES for p in P for j in J)
        operate = gp.quicksum(OMEGA[p] * OPERATE[s] * tm[s][j] * tsplit[s, r, p, j]
                              for (s, r) in NODES for p in P for j in J)
        m._z = z
    else:
        operate = gp.quicksum(OMEGA[p] * OPERATE[s] * thr[s, r, v, p]
                              for (s, r, v, p) in ACTIVE)
        m._z = None

    transport = gp.quicksum(OMEGA[p] * TRANSPORT[a, b] * flow[s, a, b, p]
                            for (s, a, b) in ARCS for p in P)
    penalty = (gp.quicksum(OMEGA[p] * PEN_SHORT * short[r, p] for r in REGIONS for p in P)
               + gp.quicksum(OMEGA[p] * pen_deviate * dev[s, r, p]
                             for (s, r) in NODES for p in P)
               + gp.quicksum(OMEGA[p] * pen_dispose * disp[r, p]
                             for r in REGIONS for p in P))

    m.setObjective(capex + operate + transport + penalty, GRB.MINIMIZE)
    m._e = dict(capex=capex, operate=operate, transport=transport, penalty=penalty)
    m._v = dict(build=build, size=size, thr=thr, flow=flow, short=short,
                dev=dev, disp=disp, cumprod=cumprod)
    m._scope = scope
    return m


_probe = build_model(learning="none")
_probe.update()
print(f"learning='none':       {_probe.NumVars:5d} vars, {_probe.NumBinVars:4d} binaries")

Set parameter Username


Set parameter LicenseID to value 2750151


Academic license - for non-commercial use only - expires 2026-12-04


learning='none':        1007 vars,   78 binaries


## 7. Calibrating the tiers

Solve without Channel B, look at how much each stage actually makes over the
horizon, and put the thresholds at $q/8$ and $q/4$ of that. The multipliers are
then exactly Wright's law per doubling.

**This is a modelling decision, not a measurement**, and it is worth being
explicit that it is arbitrary: thresholds at $q/8$ mean the fleet reaches tier 2
about two-thirds of the way through. Put them at $q/2$ and nothing ever leaves
tier 0, and the whole channel silently does nothing.

In [12]:
base = build_model(learning="none")
base.optimize()
assert base.SolCount > 0, "the calibration solve found no solution"
BASE_OBJ = base.ObjVal

prod = {}
for s in STAGES:
    prod[s] = max(base._v["cumprod"][s, rk, P[-1]].X
                  for (ss, rk) in base._scope if ss == s)

TIER_Q, TIER_M = {}, {}
for s in STAGES:
    top = max(prod[s], 1.0)
    q1 = top / 8.0
    TIER_Q[s] = [q1 * 2 ** j for j in range(N_TIERS - 1)]
    TIER_M[s] = [max(OPEX_FLOOR, (1 - LR_OPEX) ** j) for j in range(N_TIERS)]

print(f"no-learning objective : {BASE_OBJ:,.4f}")
print(f"cumulative production by stage: { {k: round(v, 1) for k, v in prod.items()} }")
print(f"tier thresholds : { {k: [round(x, 1) for x in v] for k, v in TIER_Q.items()} }")
print(f"tier multipliers: { {k: [round(x, 4) for x in v] for k, v in TIER_M.items()} }")

assert all(len(TIER_Q[s]) == N_TIERS - 1 for s in STAGES), \
    "k tiers need k-1 boundaries between them"
assert all(TIER_M[s] == sorted(TIER_M[s], reverse=True) for s in STAGES), \
    "a later tier must not be dearer than an earlier one"
print(f"\nmultipliers are exactly (1 - {LR_OPEX:.0%})^j, floored at {OPEX_FLOOR}")

no-learning objective : 45,956.7780
cumulative production by stage: {'MINE': 8132.9, 'PROC': 6994.3, 'MFG': 5993.7}
tier thresholds : {'MINE': [1016.6, 2033.2], 'PROC': [874.3, 1748.6], 'MFG': [749.2, 1498.4]}
tier multipliers: {'MINE': [1.0, 0.82, 0.6724], 'PROC': [1.0, 0.82, 0.6724], 'MFG': [1.0, 0.82, 0.6724]}

multipliers are exactly (1 - 18%)^j, floored at 0.65


## 8. Four learning variants, and whether the channels interfere

> **Predict before you run.** Channel A makes capacity cheaper; Channel B makes
> running it cheaper. Does turning both on save more, less, or exactly the sum of
> turning on each alone?

Read the **cost columns**, not the total. The total tells you which variant is
cheapest, which is not interesting; the split tells you whether each channel hit
the thing it was supposed to hit.

In [13]:
rows, res, plans = [], {}, {}
for lm in ("none", "capacity", "production", "both"):
    t0 = time.time()
    mm = build_model(learning=lm, tiers=(TIER_Q, TIER_M))
    mm.optimize()
    assert mm.SolCount > 0, f"learning={lm} found no solution"
    res[lm] = mm
    pl = {k: round(mm._v["size"][k].X, 6)
          for k in BUILD if mm._v["build"][k].X > 0.5}
    plans[lm] = tuple(sorted(pl.items()))
    rows.append(dict(learning=lm, objective=round(mm.ObjVal, 1),
                     capex=round(mm._e["capex"].getValue(), 1),
                     opex=round(mm._e["operate"].getValue(), 1),
                     builds=len(pl), capacity=round(sum(pl.values()), 1),
                     disposal=round(sum(mm._v["disp"][r, p].X
                                        for r in REGIONS for p in P), 2),
                     binaries=mm.NumBinVars, seconds=round(time.time() - t0, 1)))
variants = pd.DataFrame(rows)
variants

,learning,objective,capex,opex,builds,capacity,disposal,binaries,seconds
0,none,45956.8,9098.3,29610.8,6,1231.9,0.0,78,0.9
1,capacity,45546.0,8894.0,29544.5,6,1215.4,0.0,78,1.4
2,production,40535.3,9098.3,24153.4,6,1231.9,0.0,312,2.9
3,both,40135.6,8698.7,24153.4,6,1231.9,0.0,312,3.2


### 8.1 The separation, asserted

If the two channels were interfering — sharing a constraint they should not, or
double-counting a cost — the cleanest symptom would be Channel B moving capex, or
Channel A moving opex. Neither should happen, and the cell below requires it.

In [14]:
cap = {lm: res[lm]._e["capex"].getValue() for lm in res}
opx = {lm: res[lm]._e["operate"].getValue() for lm in res}

print(f"{'variant':11s} {'capex':>11s} {'opex':>11s}")
for lm in ("none", "capacity", "production", "both"):
    print(f"{lm:11s} {cap[lm]:11.4f} {opx[lm]:11.4f}")

assert abs(cap["production"] - cap["none"]) < 1e-6, (
    "production learning changed capex; Channel B is supposed to touch opex only")
assert abs(opx["both"] - opx["production"]) < 1e-6, (
    "adding Channel A changed Channel B's opex; the channels are interfering")
print(f"\nChannel B leaves capex untouched : {cap['none']:.4f} = {cap['production']:.4f}")
print(f"Channel A leaves Channel B's opex : {opx['production']:.4f} = {opx['both']:.4f}")
print(f"\nChannel A moves capex {cap['none']:.1f} -> {cap['capacity']:.1f} "
      f"({100 * (cap['capacity'] / cap['none'] - 1):+.2f}%)")
print(f"Channel B moves opex  {opx['none']:.1f} -> {opx['production']:.1f} "
      f"({100 * (opx['production'] / opx['none'] - 1):+.2f}%)")

variant           capex        opex
none          9098.3347  29610.8176
capacity      8893.9772  29544.4620
production    9098.3347  24153.3621
both          8698.6758  24153.3621

Channel B leaves capex untouched : 9098.3347 = 9098.3347
Channel A leaves Channel B's opex : 24153.3621 = 24153.3621

Channel A moves capex 9098.3 -> 8894.0 (-2.25%)
Channel B moves opex  29610.8 -> 24153.4 (-18.43%)


**Separable, exactly.** Channel B leaves capex at 9,098.3347 in both `none` and
`production`; Channel A leaves Channel B's opex at 24,153.3621 in both
`production` and `both`. Those are equalities to six decimal places, not
approximations, and they are the strongest single check in this notebook that the
two channels are wired to the right cost.

The sizes are very different, though. **Channel A takes 2.25% off capex; Channel
B takes 18.43% off opex** — and since opex is more than three times capex here,
production learning is worth roughly fifteen times what capacity learning is
worth. A modelling effort that implemented only Channel A, which is the more
commonly modelled one, would have found the smaller of the two effects.

## 9. Utilization — the check you should have predicted

Channel B rewards *producing*. So a plant that would otherwise idle now has a
reason to run: every unit made moves the fleet toward the next tier.

> **Predict before you run.** Does production learning push utilization up at
> every node?

In [15]:
# THE FUNCTION IS THE LESSON: utilization is computed for two models and the
# comparison is only meaningful if both are measured the same way - the same
# discipline Part 2c needed, one level down.
def utilization(mm):
    """Throughput as a share of installed capacity, per node, over the horizon."""
    thr, size = mm._v["thr"], mm._v["size"]
    out = {}
    for (s, r) in NODES:
        used = cap_ = 0.0
        for p in P:
            for v in VIN[s, r, p]:
                used += LEN[p] * thr[s, r, v, p].X
                cap_ += LEN[p] * (LEGACY_CAP[s, r] if v == -1 else size[s, r, v].X)
        out[s, r] = 100.0 * used / cap_ if cap_ > 0 else 0.0
    return out


un, up = utilization(res["none"]), utilization(res["production"])
util = pd.DataFrame([dict(node=f"{s}/{r}", none=round(un[s, r], 1),
                          production=round(up[s, r], 1),
                          change=round(up[s, r] - un[s, r], 1))
                     for (s, r) in NODES])
assert all(v > 50 for v in un.values()), "a node is idling badly even without learning"
util

,node,none,production,change
0,MINE/R1,96.0,95.3,-0.7
1,MINE/R2,89.6,90.4,0.8
2,PROC/R1,97.6,96.9,-0.7
3,PROC/R2,95.9,96.7,0.8
4,MFG/R1,76.3,75.8,-0.5
5,MFG/R2,95.8,96.6,0.8


**Not uniformly, and that is the interesting part.** Utilization rises at three
nodes and falls at three. Production learning does not simply say "run
everything harder" — it says "run the nodes whose next tier is within reach", and
for a node that cannot get there in time the optimal answer is to let a
better-placed node do the producing.

MFG/R1 is the low one at 76% in both cases. That is not a defect: R1's
manufacturing sits next to the larger legacy fleet, so it has capacity it does
not need early on, and the model would rather leave it idle than build less of it
and fail demand later.

## 10. Which tiers actually activate?

A tier structure that never leaves tier 0 is decoration. A tier structure that
jumps straight to the last tier is a threshold set too low. Neither raises an
error, so look.

In [16]:
mz = res["production"]
tiers_path = []
for (s, rk) in mz._scope:
    path = [next(j for j in range(N_TIERS) if mz._z[s, rk, p, j].X > 0.5) for p in P]
    tiers_path.append(dict(stage=s, scope=rk, tier_by_period=path,
                           cumprod_end=round(mz._v["cumprod"][s, rk, P[-1]].X, 1)))
    assert path == sorted(path), (
        f"{s}/{rk} went BACKWARDS through the tiers; a cumulative driver cannot "
        f"decrease, so this is a constraint error")
    assert path[0] == 0, f"{s}/{rk} started above tier 0"
tier_table = pd.DataFrame(tiers_path)
assert max(max(r["tier_by_period"]) for r in tiers_path) == N_TIERS - 1, \
    "no node ever reached the top tier; the thresholds are too high to bind"
print("every node progresses 0 -> 1 -> 2 monotonically, and every node gets there")
tier_table

every node progresses 0 -> 1 -> 2 monotonically, and every node gets there


,stage,scope,tier_by_period,cumprod_end
0,MINE,R1,"[0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2]",8074.8
1,MINE,R2,"[0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2]",6947.0
2,PROC,R1,"[0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2]",6944.3
3,PROC,R2,"[0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2]",5974.4
4,MFG,R1,"[0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2]",5955.0
5,MFG,R2,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2]",4721.1


Monotone everywhere, which a cumulative driver requires and the assertion checks.
Every node reaches the top tier, and MFG/R2 gets there one period later than the
rest — it is the smallest node, so it takes longest to accumulate.

The transition happens around period 7–8, roughly year 13. With a three-year lag
that means the production justifying it happened around year 10, which is when
the first wave of new capacity has been running for a while. The timing is a
consequence of the calibration in §7, not an independent finding.

## 11. Would a planner overproduce just to learn faster?

This is the question a production-learning model exists to be asked. Making more
than you need is wasteful — but it moves you down the learning curve, and the
discount applies to everything you make afterwards. A model that gets this wrong
in the permissive direction will happily manufacture and bin product forever.

The lever is `PEN_DISPOSE`. Lower it to zero and disposal is free.

> **Predict before you run.** With free disposal, does the planner overproduce?

In [17]:
rows = []
for pen in (12.0, 6.0, 3.0, 1.0, 0.0):
    mm = build_model(learning="production", tiers=(TIER_Q, TIER_M), pen_dispose=pen)
    mm.optimize()
    rows.append(dict(disposal_penalty=pen, objective=round(mm.ObjVal, 1),
                     disposal_units=round(sum(mm._v["disp"][r, p].X
                                              for r in REGIONS for p in P), 2)))
dump = pd.DataFrame(rows)
assert (dump.disposal_units < 1e-6).all(), \
    "the planner disposed of product; section 11's conclusion needs rewriting"
print("zero disposal at every penalty, INCLUDING free disposal")
dump

zero disposal at every penalty, INCLUDING free disposal


,disposal_penalty,objective,disposal_units
0,12.0,40535.3,0.0
1,6.0,40535.3,0.0
2,3.0,40535.3,0.0
3,1.0,40535.3,0.0
4,0.0,40535.3,0.0


Zero, everywhere, including at a penalty of exactly zero. Worth pushing harder
before believing it — the learning rate might simply be too weak to be worth
gaming. So: crank it, and drop the floor so the higher rate can actually bite.

In [18]:
rows = []
for lr, floor in ((0.18, 0.65), (0.35, 0.25), (0.55, 0.25)):
    tq = {s: list(TIER_Q[s]) for s in STAGES}      # thresholds unchanged
    tm = {s: [max(floor, (1 - lr) ** j) for j in range(N_TIERS)] for s in STAGES}
    mm = build_model(learning="production", tiers=(tq, tm), pen_dispose=0.0)
    mm.optimize()
    rows.append(dict(LR_opex=lr, floor=floor,
                     multipliers=[round(x, 3) for x in tm["PROC"]],
                     objective=round(mm.ObjVal, 1),
                     disposal_units=round(sum(mm._v["disp"][r, p].X
                                              for r in REGIONS for p in P), 2)))
hard = pd.DataFrame(rows)
assert (hard.disposal_units < 1e-6).all(), \
    "disposal appeared under an aggressive learning rate; the section is wrong"
print("still zero, at a 55% learning rate with free disposal")
hard

still zero, at a 55% learning rate with free disposal


,LR_opex,floor,multipliers,objective,disposal_units
0,0.18,0.65,"[1.0, 0.82, 0.672]",40535.3,0.0
1,0.35,0.25,"[1.0, 0.65, 0.423]",36261.2,0.0
2,0.55,0.25,"[1.0, 0.45, 0.25]",33030.0,0.0


**Still zero.** The objective falls a long way — 40,535.3 down to 33,030.0 as the
rate goes from 18% to 55% — so the channel is certainly doing something. It is
just never worth *manufacturing waste* to get there.

The reason is structural and worth naming: the tiers are driven by cumulative
production, and the model is already producing near capacity to serve demand.
Buying an earlier tier transition would mean building **more capacity** and
running it to make product nobody wants. The capex of that extra capacity
outweighs the opex discount it unlocks, at every rate tested.

**That is a result about this instance, not a theorem.** A model with cheaper
capacity, a steeper curve, or a longer horizon over which to amortise the
discount could easily flip it. What the section demonstrates is the *test*, not
the answer — and note that the negative result required removing the penalty
entirely to be convincing. A conclusion drawn at `PEN_DISPOSE = 12` would have
proved only that disposal is expensive.

## 12. The government lever: local content minimums

A minimum throughput at every node, phased in from year 6. This is the one place
in the notebook where a *non-market* constraint drives behaviour, and it is where
the disposal mechanism finally earns its place in the model.

In [19]:
rows = []
for level in (0.0, 60.0, 110.0, 160.0):
    tmin = ({} if level <= 0 else
            {(s, r, p): (0.0 if START[p] < TIER_MIN_PHASE_IN else level)
             for (s, r) in NODES for p in P})
    mm = build_model(learning="production", tiers=(TIER_Q, TIER_M), tier_min=tmin)
    mm.optimize()
    pl = {k for k in BUILD if mm._v["build"][k].X > 0.5}
    rows.append(dict(min_throughput=level, objective=round(mm.ObjVal, 1),
                     undersupply=round(sum(mm._v["dev"][s, r, p].X
                                           for (s, r) in NODES for p in P), 2),
                     disposal=round(sum(mm._v["disp"][r, p].X
                                        for r in REGIONS for p in P), 2),
                     builds=len(pl)))
lcr = pd.DataFrame(rows)
assert lcr.disposal.iloc[-1] > 1, (
    "the strictest local-content level produced no disposal, so this section's "
    "point about forced overproduction has gone")
lcr

,min_throughput,objective,undersupply,disposal,builds
0,0.0,40535.3,0.0,0.00,6
1,60.0,40535.3,0.0,0.00,6
2,110.0,40558.6,0.0,0.00,6
3,160.0,52255.8,0.0,260.46,8


**Here is where disposal activates, and it validates the mechanism.** At a floor
of 160 the planner builds two extra facilities, produces 260.46 units it cannot
sell, and pays to dispose of them — pushing the objective from 40,535.3 to
52,255.8, a 29% increase.

That is the difference between the two sections. §11 asked whether a planner
would *choose* to overproduce for a commercial reason and the answer was no, at
every rate. §12 shows what it looks like when a planner is *made* to overproduce
by a constraint that ignores demand. The disposal variable was never dead code —
it was waiting for a policy that would make it bind.

Note the step: 60 and 110 cost almost nothing (40,535.3 and 40,558.6), then 160
costs 29%. Local-content rules are cheap while they sit below what the chain was
going to do anyway, and expensive the moment they exceed it. A policy set at 110
here would look free and be one unit of demand growth away from being ruinous.

## 13. Does any of this change the *plan*?

Part 3 found that four accounting and learning variants gave one identical build
plan. The same question here, with two genuine learning channels rather than
one.

In [20]:
n_distinct = len(set(plans.values()))
comp = pd.DataFrame([
    dict(learning=lm, builds=len(plans[lm]),
         total_capacity=round(sum(v for _k, v in plans[lm]), 1),
         mean_size=round(sum(v for _k, v in plans[lm]) / len(plans[lm]), 1),
         build_years=sorted(START[v] for ((_s, _r, v), _x) in plans[lm]))
    for lm in ("none", "capacity", "production", "both")])

print(f"DISTINCT BUILD PLANS among the four variants: {n_distinct}")
assert plans["none"] == plans["production"] == plans["both"], (
    "production learning changed the build plan; section 13's claim is that it "
    "does not, so the prose needs rewriting rather than the assertion relaxing")
assert plans["capacity"] != plans["none"], \
    "capacity learning changed nothing at all, which section 13 says it does"
print("`none`, `production` and `both` share a plan; only `capacity` differs")
comp

DISTINCT BUILD PLANS among the four variants: 2
`none`, `production` and `both` share a plan; only `capacity` differs


,learning,builds,total_capacity,mean_size,build_years
0,none,6,1231.9,205.3,"[7, 7, 10, 13, 19, 24]"
1,capacity,6,1215.4,202.6,"[7, 7, 10, 13, 19, 19]"
2,production,6,1231.9,205.3,"[7, 7, 10, 13, 19, 24]"
3,both,6,1231.9,205.3,"[7, 7, 10, 13, 19, 24]"


**Two plans out of four variants**, and the split is instructive.

`none`, `production` and `both` build the same six facilities in the same years
— [7, 7, 10, 13, 19, 24]. Production learning cuts opex by 18% and does not move
a single build decision, because it makes *running* the fleet cheaper and the
fleet you want is determined by demand.

Only Channel A moves anything, and barely: it pulls the last build from year 24
to **year 19** and trims total capacity from 1,231.9 to 1,215.4. That is exactly
what capacity learning should do — building earlier is now worth slightly more,
because it makes the next build cheaper.

**So the honest summary is that both channels are worth a lot of money and
almost no decisions.** That is a legitimate and common finding, and the only way
to know it is to compare plans. It also means a modeller who needs the *plan*
and not the *cost* could skip Channel B entirely here — and one who needs the
cost cannot.

## 14. The agreement assertion

`src/lithium/netcore.py` holds the same model, and the same one covers Part 3:
`build_netcore` with `learning='capacity'` and no tiers *is* Part 3, and adding
tiers gives this notebook. One implementation, two notebooks — the same
arrangement `add_region` uses for Parts 4c and 4e.

This compares the tier calibration, all four variants' objectives, their **cost
components separately** (which is what would catch a channel wired to the wrong
term), and their build plans.

In [21]:
from lithium import NetCoreInstance, build_netcore_structure
from lithium import curves as pkg_curves
from lithium import netcore as NC

nb_inst = NetCoreInstance(
    stages=STAGES, regions=REGIONS, fixed=FIXED, unit=UNIT, operate=OPERATE,
    lead=LEAD, legacy_cap=LEGACY_CAP, legacy_ret=LEGACY_RET,
    demand_base=DEMAND_BASE, demand_growth=DEMAND_GROWTH,
    eta_ceil=ETA_CEIL, eta_base=ETA_BASE, alpha=ALPHA, beta=BETA,
    delta_bar=DELTA_BAR)
nb_st = build_netcore_structure(
    nb_inst, blocks=BLOCKS, dr=DR, life=LIFE, cap_min=CAP_MIN, cap_max=CAP_MAX,
    legacy_byr=LEGACY_BYR, eta_floor=ETA_FLOOR,
    transport_own=TRANSPORT_OWN, transport_cross=TRANSPORT_CROSS)

pkg_QBP, pkg_CBPm = pkg_curves.capex_breakpoints(
    Q_START, Q_ADD, NBP, LR_CAPEX, CAPEX_FLOOR, panels=PANELS)
pkg_CBP = [U0 * c for c in pkg_CBPm]
assert max(abs(a - b) for a, b in zip(CBP, pkg_CBP)) / max(CBP) < 1e-12, \
    "the hand-built capex curve and lithium.curves differ"

(ptq, ptm), pkg_base, pkg_prod = NC.calibrate_tiers(
    nb_st, n_tiers=N_TIERS, lr_opex=LR_OPEX, opex_floor=OPEX_FLOOR,
    capex_curve=(pkg_QBP, pkg_CBP), learn_stages=tuple(LEARN_STAGES),
    pen_short=PEN_SHORT, pen_dispose=PEN_DISPOSE, pen_deviate=PEN_DEVIATE,
    allow_dispose=True, mipgap=MIPGAP)
rel = abs(pkg_base - BASE_OBJ) / abs(BASE_OBJ)
print(f"{'calibration objective':28s} notebook {BASE_OBJ:12.4f}  "
      f"package {pkg_base:12.4f}  rel {rel:.1e}")
assert rel < 1e-9, f"the calibration solves disagree by {rel:.2e}"
for s in STAGES:
    assert max(abs(a - b) for a, b in zip(TIER_Q[s], ptq[s])) < 1e-6
    assert max(abs(a - b) for a, b in zip(TIER_M[s], ptm[s])) < 1e-12
print(f"{'tier thresholds and multipliers':28s} identical for all "
      f"{len(STAGES)} stages")

calibration objective        notebook   45956.7780  package   45956.7780  rel 0.0e+00
tier thresholds and multipliers identical for all 3 stages


The calibration agreeing is the precondition; the models themselves are what
matter. This compares each variant's objective, **every cost component
separately** — which is what would catch a channel wired to the wrong term —
and the build plan.

In [22]:
print(f"{'variant':11s} {'notebook':>12s} {'package':>12s} {'rel':>9s}  "
      f"components  plan")
for lm in ("none", "capacity", "production", "both"):
    a = res[lm]
    b = NC.solve_netcore(nb_st, learning=lm, capex_curve=(pkg_QBP, pkg_CBP),
                         tiers=(ptq, ptm), learn_stages=tuple(LEARN_STAGES),
                         learn_scope=LEARN_SCOPE, n_tiers=N_TIERS,
                         lag_years=LAG_YEARS, pen_short=PEN_SHORT,
                         pen_dispose=PEN_DISPOSE, pen_deviate=PEN_DEVIATE,
                         allow_dispose=True, mipgap=MIPGAP)
    rel = abs(a.ObjVal - b["obj"]) / abs(b["obj"])
    worst_c = max(abs(a._e[k].getValue() - b["components"][k])
                  / max(abs(b["components"][k]), 1.0)
                  for k in ("capex", "operate", "transport", "penalty"))
    nb_plan = {k: round(a._v["size"][k].X, 6)
               for k in BUILD if a._v["build"][k].X > 0.5}
    same = nb_plan == b["plan"]
    print(f"{lm:11s} {a.ObjVal:12.4f} {b['obj']:12.4f} {rel:9.1e}  "
          f"{worst_c:10.1e}  {'same' if same else '** DIFFERS **'}")
    assert rel < 1e-9, f"{lm}: objectives disagree by {rel:.2e}"
    assert worst_c < 1e-9, f"{lm}: a cost component disagrees by {worst_c:.2e}"
    assert same, f"{lm}: same objective, different build plan"

print("\nnotebook and package agree on the calibration, the tiers, all four")
print("objectives, every cost component separately, and all four build plans")

variant         notebook      package       rel  components  plan


none          45956.7780   45956.7780   0.0e+00     0.0e+00  same


capacity      45546.0104   45546.0104   9.6e-16     4.2e-15  same


production    40535.2925   40535.2925   0.0e+00     0.0e+00  same


both          40135.6335   40135.6335   0.0e+00     7.5e-16  same

notebook and package agree on the calibration, the tiers, all four
objectives, every cost component separately, and all four build plans


## 15. Summary

| Question | Answer |
|---|---|
| Do the two channels interfere? | **No** — capex identical under `none`/`production`, opex identical under `production`/`both` |
| Which is worth more? | Channel B, by roughly 15× — opex −18.43% against capex −2.25% |
| Does production learning change the plan? | **No.** Same six builds, same years |
| Does capacity learning? | Barely — one build moves from year 24 to 19 |
| Would a planner overproduce to learn faster? | **No**, at any disposal penalty down to zero and any rate up to 55% |
| When does disposal ever happen? | Only when a local-content floor forces it: 260.46 units at a floor of 160 |
| What does a 160 local-content floor cost? | **+29%**, against ~0% at 110 |

### Formulation lessons

- **A learning channel that multiplies a variable multiplier by a variable
  quantity is bilinear.** Tiers plus a throughput split make it linear again, for
  234 binaries.
- **Define a lag in the unit it means.** Three *years* mapped onto periods stays
  three years; three *periods* silently becomes nine years once the mesh coarsens.
- **A knowledge stock is not discounted.** `cumprod` uses `LEN`, never `OMEGA` —
  it counts units made, not their present value.
- **Split throughput at the coarsest index the rate depends on.** The opex rate
  is vintage-independent, so splitting node-level throughput is exactly
  equivalent to splitting per-vintage and much smaller.
- **Test a negative result by removing the obstacle entirely.** "No disposal at a
  penalty of 12" proves nothing; "no disposal at a penalty of 0, at a 55%
  learning rate" is a finding.
- **Compare plans, not just objectives** — the same lesson Part 3 ends on, and
  here it separates a channel worth 18% of opex from one that changes what you
  build.

### Things to try

- `LAG_YEARS = 0` — instant know-how, and watch the tier transitions jump earlier
- `q1 = top / 2.0` in section 7 — thresholds so high nothing leaves tier 0, and
  section 10's assertion catching it
- `LEARN_SCOPE = 'global'` — one industry-wide production pool instead of
  per-region, so R2 free-rides on R1's experience
- `N_TIERS = 6` — a finer approximation to a smooth curve, and the binary count
  it costs
- `TIER_MIN_PHASE_IN = 1` — a local-content floor from day one, which the legacy
  fleet cannot satisfy

### Where this goes next

**Part 4** takes this chain and hands it to two firms who do not cooperate. The
planner's question — what should be built — becomes an equilibrium question, and
the learning pool that is industry-wide here becomes firm-specific, which turns
out to change the answer far more than either channel here did.